# 00 - Preparar el proyecto

Este notebook deja todo listo para que los demás funcionen. Hace cuatro cosas:

1. Le dice a Python dónde está nuestro código (la carpeta `src/bondis`), para poder escribir `from bondis import ...`.
2. Crea en Databricks el lugar donde se guardan las tablas y los archivos (`bondis_stm.datos.raw`, definido en `config.py`).
3. Crea una carpeta para cada tipo de archivo (viajes, clima, etc.).
4. Muestra cómo dividimos los datos en train, val y test.

Se puede correr todas las veces que quieras. Si algo ya existe, no lo vuelve a crear ni lo borra.

No hace falta abrirlo a mano: los otros notebooks lo corren al principio con `%run ./00_setup`.

## 1. Encontrar nuestro código

Python solo puede importar código de las carpetas que conoce. Esta celda busca la carpeta del repositorio subiendo carpeta por carpeta hasta encontrar `src/bondis`, y la agrega a la lista de carpetas que Python conoce (`sys.path`).

In [0]:
import importlib
import os
import sys
 
def _buscar_raiz_repo(inicio):
    """Sube carpeta por carpeta desde 'inicio' hasta encontrar src/bondis.
 
    Así funciona igual si el notebook se corre solo o si se llama con %run desde otro.
    """
    actual = os.path.abspath(inicio)
    while True:
        if os.path.isdir(os.path.join(actual, "src", "bondis")):
            return actual
        padre = os.path.dirname(actual)
        if padre == actual:
            return None
        actual = padre
 
 
_contexto = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_notebook_path = "/Workspace" + _contexto.notebookPath().get()
 
REPO_ROOT = _buscar_raiz_repo(os.getcwd()) or _buscar_raiz_repo(os.path.dirname(_notebook_path))
if REPO_ROOT is None:
    raise FileNotFoundError(
        "No se encontró la carpeta src/bondis.\n"
        f"  Directorio de trabajo: {os.getcwd()}\n"
        f"  Notebook: {_notebook_path}\n"
        "Verificar que el notebook esté dentro del Git folder del repo."
    )
SRC_PATH = os.path.join(REPO_ROOT, "src")
 
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)
 
from bondis import config
 
importlib.reload(config)  # vuelve a leer config.py por si lo cambiamos, sin reiniciar el cluster
 
print("Raíz del repo:      ", REPO_ROOT)
print("Ruta de código:     ", SRC_PATH)

## 2. Crear el catálogo, el esquema y el volume

Databricks ordena todo en tres niveles, como carpetas una dentro de otra:

- **Catálogo** (`bondis_stm`): la carpeta grande del proyecto.
- **Esquema** (`datos`): una subcarpeta donde van las tablas.
- **Volume** (`raw`): una carpeta para guardar archivos sueltos, como los ZIP y CSV que bajamos.


In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {config.CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config.CATALOG}.{config.SCHEMA}")
spark.sql(
    f"CREATE VOLUME IF NOT EXISTS {config.CATALOG}.{config.SCHEMA}.{config.VOLUME}"
)
 
print(f"Catálogo y esquema listos: {config.CATALOG}.{config.SCHEMA}")
print(f"Volume listo:  {config.RAW_PATH}")

## 3. Crear una carpeta para cada tipo de archivo

| Carpeta | Qué va adentro |
|---|---|
| `viajes_zip/` | Los ZIP de viajes de cada mes, bajados a mano del portal de la Intendencia |
| `viajes/` | Los CSV de viajes, ya sacados de los ZIP |
| `clima/` | Lluvia y temperatura de cada hora |
| `calendario/` | Por ahora vacía. El calendario está escrito en `src/bondis/calendario.py` |
| `paros/` | Días y horas con paro (todavía no lo hicimos) |

In [0]:
for carpeta in [config.RAW_ZIP, config.RAW_VIAJES, config.RAW_CLIMA, config.RAW_CALENDARIO, config.RAW_PAROS]:
    dbutils.fs.mkdirs(carpeta)
    print("Carpeta lista:", carpeta)

## 4. Revisar que todo esté bien

Muestra la versión de Spark, el usuario y cuántos archivos hay en cada carpeta. Sirve para ver de un vistazo si falta subir algo.

In [0]:
print("Versión de Spark:", spark.version)
print("Usuario actual:  ", spark.sql("SELECT current_user()").first()[0])
print()
 
for carpeta in [config.RAW_ZIP, config.RAW_VIAJES, config.RAW_CLIMA, config.RAW_CALENDARIO, config.RAW_PAROS]:
    archivos = [f for f in dbutils.fs.ls(carpeta) if not f.isDir()]
    total_mb = sum(f.size for f in archivos) / 1e6
    print(f"{carpeta}: {len(archivos)} archivo(s), {total_mb:.1f} MB")
    for f in archivos:
        print(f"    - {f.name}  ({f.size / 1e6:.1f} MB)")

In [0]:
display(spark.sql(f"SHOW TABLES IN {config.CATALOG}.{config.SCHEMA}"))

## 5. Cómo dividimos los datos

Usamos las fechas para dividir, no el azar, porque queremos predecir el futuro con el pasado:

| Conjunto | Fechas | Para qué se usa |
|---|---|---|
| `train` | enero a setiembre de 2024 | El modelo aprende con estos datos |
| `val` | octubre a diciembre de 2024 | Comparamos modelos y elegimos el mejor |
| `test` | todo 2025 | Probamos el modelo elegido una sola vez, al final |

La fecha de fin **no** está incluida. Por eso siempre se filtra con `fecha >= inicio` y `fecha < fin`.

`train_final` es train + val juntos. Cuando ya elegimos el modelo, lo volvemos a entrenar con todo 2024 antes de probarlo en test.

In [0]:
print(f"{'Conjunto':<12} {'Inicio':<12} {'Fin (excl.)':<12}")
for nombre, (inicio, fin) in config.PARTICIONES.items():
    print(f"{nombre:<12} {inicio:<12} {fin:<12}")
print(f"{'train_final':<12} {config.PARTICION_TRAIN_FINAL[0]:<12} {config.PARTICION_TRAIN_FINAL[1]:<12}")
 
print()
print(f"Horizonte de predicción: {config.HORIZONTE_DIAS} día(s)")
print(f"Semilla: {config.SEED}")